In [21]:
from mcp.server import MCPServer

mcp = MCPServer("Travel MCP Server")

print("MCP Server created")

MCP Server created


In [22]:
@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [23]:
@mcp.tool()
def search_flights(
    origin: str,
    destination: str
) -> str:
    """Search available flights between two cities."""

    return (
        f"Available flights found from "
        f"{origin} to {destination}"
    )


In [24]:
@mcp.tool()
def book_hotel(
    city: str,
    nights: int
) -> str:
    """Book a hotel in a city."""

    return (
        f"Hotel booked in {city} "
        f"for {nights} nights."
    )

In [25]:
@mcp.resource("travel://preferences")
def travel_preferences() -> str:
    """Return user's travel preferences."""

    return """
Preferred airline: Emirates
Preferred seat: Window
Preferred class: Economy
Preferred hotel: 4 Star
"""

travel://preferences
        ↓
Travel preference data

In [ ]:
@mcp.resource("weather://{city}")
def weather(city: str) -> str:
    """Return weather information for a city."""

    return f"Weather information for {city}"

In [ ]:
@mcp.prompt()
def plan_vacation(
    destination: str,
    days: str
) -> str:
    """Create instructions for planning a vacation."""

    return f"""
Plan a {days}-day vacation to {destination}.

Check:
1. Travel preferences
2. Available flights
3. Weather
4. Hotel options
"""

Travel MCP Server
│
├── Tools
│   ├── add()
│   ├── search_flights()
│   └── book_hotel()
│
├── Resources
│   ├── travel://preferences
│   └── weather://{city}
│
└── Prompts
    └── plan_vacation()

In [26]:
from mcp import Client


async def show_server_details():

    async with Client(mcp) as client:

        print("Protocol Version:")
        print(client.protocol_version)

        print("\nServer Info:")
        print(client.server_info)

        print("\nServer Capabilities:")
        print(client.server_capabilities)

In [27]:
await show_server_details()

Protocol Version:
2026-07-28

Server Info:
name='Travel MCP Server' title=None version='' description=None website_url=None icons=None

Server Capabilities:
experimental=None logging=None prompts=PromptsCapability(list_changed=True) resources=ResourcesCapability(subscribe=True, list_changed=True) tools=ToolsCapability(list_changed=True) completions=None extensions=None tasks=None


In [28]:
async def show_tools():

    async with Client(mcp) as client:

        tools_result = await client.list_tools()

        print("Available Tools:\n")

        for tool in tools_result.tools:
            print("Name:", tool.name)
            print("Description:", tool.description)
            print("Input Schema:", tool.input_schema)
            print("-" * 50)


In [29]:
await show_tools()

Available Tools:

Name: add
Description: Add two numbers.
Input Schema: {'type': 'object', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'addArguments'}
--------------------------------------------------
Name: search_flights
Description: Search available flights between two cities.
Input Schema: {'type': 'object', 'properties': {'origin': {'title': 'Origin', 'type': 'string'}, 'destination': {'title': 'Destination', 'type': 'string'}}, 'required': ['origin', 'destination'], 'title': 'search_flightsArguments'}
--------------------------------------------------
Name: book_hotel
Description: Book a hotel in a city.
Input Schema: {'type': 'object', 'properties': {'city': {'title': 'City', 'type': 'string'}, 'nights': {'title': 'Nights', 'type': 'integer'}}, 'required': ['city', 'nights'], 'title': 'book_hotelArguments'}
--------------------------------------------------


In [32]:
from mcp.types import TextContent


async def call_add_tool():

    async with Client(mcp) as client:

        result = await client.call_tool(
            "add",
            {
                "a": 10,
                "b": 20
            }
        )

        print("Tool Error:", result.is_error)

        print("\nTool Result:")

        for block in result.content:
            if isinstance(block, TextContent):
                print(block.text)


await call_add_tool()

Tool Error: False

Tool Result:
30


MCP Client
    ↓
tools/call
    ↓
MCP Server
    ↓
add(a=10, b=20)
    ↓
30
    ↓
MCP Client

In [33]:
async def call_flight_tool():

    async with Client(mcp) as client:

        result = await client.call_tool(
            "search_flights",
            {
                "origin": "Bangalore",
                "destination": "Dubai"
            }
        )

        for block in result.content:
            if isinstance(block, TextContent):
                print(block.text)

In [34]:
await call_flight_tool()

Available flights found from Bangalore to Dubai


In [35]:
async def show_resources():

    async with Client(mcp) as client:

        # Fixed resources
        resources = await client.list_resources()

        print("Direct Resources:")
        for resource in resources.resources:
            print(resource.uri)

        # Dynamic resource templates
        templates = await client.list_resource_templates()

        print("\nResource Templates:")
        for template in templates.resource_templates:
            print(template.uri_template)

In [36]:
await show_resources()

Direct Resources:
travel://preferences

Resource Templates:


In [37]:
from mcp.types import TextResourceContents


async def read_preferences():

    async with Client(mcp) as client:

        result = await client.read_resource(
            "travel://preferences"
        )

        for content in result.contents:
            if isinstance(content, TextResourceContents):
                print(content.text)


await read_preferences()


Preferred airline: Emirates
Preferred seat: Window
Preferred class: Economy
Preferred hotel: 4 Star



In [39]:
async def read_weather():

    async with Client(mcp) as client:

        result = await client.read_resource(
            "weather://Barcelona"
        )

        for content in result.contents:
            if isinstance(content, TextResourceContents):
                print(content.text)


await read_weather()

  + Exception Group Traceback (most recent call last):
  |   File "d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\IPython\core\interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\Sunny\AppData\Local\Temp\ipykernel_34848\928512098.py", line 14, in <module>
  |     await read_weather()
  |   File "C:\Users\Sunny\AppData\Local\Temp\ipykernel_34848\928512098.py", line 3, in read_weather
  |     async with Client(mcp) as client:
  |   File "C:\Users\Sunny\AppData\Roaming\uv\python\cpython-3.11.13-windows-x86_64-none\Lib\contextlib.py", line 745, in __aexit__
  |     raise exc_details[1]
  |   File "C:\Users\Sunny\AppData\Roaming\uv\python\cpython-3.11.13-windows-x86_64-none\Lib\contextlib.py", line 728, in __aexit__
  |     cb_suppress = await cb(*exc_details)
  |                   ^^^^^^^^^^^^^^^^^^^^^^
  |   File "d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-pac

In [40]:
async def show_prompts():

    async with Client(mcp) as client:

        result = await client.list_prompts()

        print("Available Prompts:")

        for prompt in result.prompts:
            print(prompt.name)
            print(prompt.description)
            print(prompt.arguments)
            print("-" * 40)


await show_prompts()

Available Prompts:


In [41]:
async def get_vacation_prompt():

    async with Client(mcp) as client:

        result = await client.get_prompt(
            "plan_vacation",
            {
                "destination": "Barcelona",
                "days": "7"
            }
        )

        for message in result.messages:

            print("Role:", message.role)

            if isinstance(message.content, TextContent):
                print(message.content.text)


await get_vacation_prompt()

[08/12/26 17:01:37] ERROR    Error getting prompt plan_vacation                                      ]8;id=8722805;file://d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\mcp\server\mcpserver\server.py\server.py]8;;\:]8;id=8722806;file://d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\mcp\server\mcpserver\server.py#1296\1296]8;;\
                             ╭───────────────── Traceback (most recent call last) ─────────────────╮               
                             │ d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site- │               
                             │ packages\mcp\server\mcpserver\server.py:1283 in get_prompt          │               
                             │                                                                     │               
                             │   1280 │   │   try:                                                 │               
                             │   1281 │   │   │   prompt = self._prompt_manager.get_prompt(name)   │               
                             │   1282 │   │   │   if not prompt:                                   │               
                             │ ❱ 1283 │   │   │   │   raise ValueError(f"Unknown prompt: {name}")  │               
                             │   1284 │   │   │                                                    │               
                             │   1285 │   │   │   rendered = await prompt.render(arguments, contex │               
                             │   1286 │   │   │   if isinstance(rendered, InputRequiredResult):    │               
                             ╰─────────────────────────────────────────────────────────────────────╯               
                             ValueError: Unknown prompt: plan_vacation                                             

[08/12/26 17:01:38] ERROR    request handler raised                                        ]8;id=8722813;file://d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\mcp\shared\direct_dispatcher.py\direct_dispatcher.py]8;;\:]8;id=8722814;file://d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\mcp\shared\direct_dispatcher.py#280\280]8;;\
                             ╭──────────── Traceback (most recent call last) ────────────╮                         
                             │ d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env │                         
                             │ \Lib\site-packages\mcp\server\mcpserver\server.py:1283 in │                         
                             │ get_prompt                                                │                         
                             │                                                           │                         
                             │   1280 │   │   try:                                       │                         
                             │   1281 │   │   │   prompt = self._prompt_manager.get_prom │                         
                             │   1282 │   │   │   if not prompt:                         │                         
                             │ ❱ 1283 │   │   │   │   raise ValueError(f"Unknown prompt: │                         
                             │   1284 │   │   │                                          │                         
                             │   1285 │   │   │   rendered = await prompt.render(argumen │                         
                             │   1286 │   │   │   if isinstance(rendered, InputRequiredR │                         
                             ╰───────────────────────────────────────────────────────────╯                         
                             ValueError: Unknown prompt: plan_vacation                                             
                                                                                                                   
                             The above exception was the direct cause of the following                             
                             exception:                                                                            
                                                                                                                   
                             ╭──────────── Traceback (most recent call last) ────────────╮                         
                             │ d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env │                         
                             │ \Lib\site-packages\mcp\shared\direct_dispatcher.py:266 in │                         
                             │ _dispatch_request                                         │                         
                             │                                                           │                         
                             │   263 │   │   │   │   self._in_flight_ids.add(in_flight_k │                         
                             │   264 │   │   │   │   dctx = self._make_context(on_progre │                         
                             │       request_id=request_id)                              │                         
                             │   265 │   │   │   │   try:                                │                         
                             │ ❱ 266 │   │   │   │   │   return await self._on_request(d │                         
                             │   267 │   │   │   │   except MCPError:                    │                         
                             │   268 │   │   │   │   │   raise                           │                         
                             │   269 │   │   │   │   except ValidationError as e:        │                         
        

  + Exception Group Traceback (most recent call last):
  |   File "d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\IPython\core\interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\Sunny\AppData\Local\Temp\ipykernel_34848\3019981863.py", line 21, in <module>
  |     await get_vacation_prompt()
  |   File "C:\Users\Sunny\AppData\Local\Temp\ipykernel_34848\3019981863.py", line 3, in get_vacation_prompt
  |     async with Client(mcp) as client:
  |   File "C:\Users\Sunny\AppData\Roaming\uv\python\cpython-3.11.13-windows-x86_64-none\Lib\contextlib.py", line 745, in __aexit__
  |     raise exc_details[1]
  |   File "C:\Users\Sunny\AppData\Roaming\uv\python\cpython-3.11.13-windows-x86_64-none\Lib\contextlib.py", line 728, in __aexit__
  |     cb_suppress = await cb(*exc_details)
  |                   ^^^^^^^^^^^^^^^^^^^^^^
  |   File "d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\

In [42]:
async def complete_demo():

    async with Client(mcp) as client:

        # ---------------------------------
        # 1. MCP CONNECTION
        # ---------------------------------

        print("=" * 60)
        print("CONNECTED TO MCP SERVER")
        print("=" * 60)

        print("Protocol:", client.protocol_version)
        print("Server:", client.server_info)


        # ---------------------------------
        # 2. DISCOVER TOOLS
        # ---------------------------------

        print("\n" + "=" * 60)
        print("AVAILABLE TOOLS")
        print("=" * 60)

        tools = await client.list_tools()

        for tool in tools.tools:
            print("-", tool.name)


        # ---------------------------------
        # 3. CALL TOOL
        # ---------------------------------

        print("\n" + "=" * 60)
        print("CALLING TOOL")
        print("=" * 60)

        result = await client.call_tool(
            "search_flights",
            {
                "origin": "Bangalore",
                "destination": "Barcelona"
            }
        )

        for block in result.content:
            if isinstance(block, TextContent):
                print(block.text)


        # ---------------------------------
        # 4. LIST RESOURCES
        # ---------------------------------

        print("\n" + "=" * 60)
        print("RESOURCES")
        print("=" * 60)

        resources = await client.list_resources()

        for resource in resources.resources:
            print("-", resource.uri)


        # ---------------------------------
        # 5. READ RESOURCE
        # ---------------------------------

        print("\n" + "=" * 60)
        print("TRAVEL PREFERENCES")
        print("=" * 60)

        resource_result = await client.read_resource(
            "travel://preferences"
        )

        for content in resource_result.contents:
            if isinstance(content, TextResourceContents):
                print(content.text)


        # ---------------------------------
        # 6. RESOURCE TEMPLATE
        # ---------------------------------

        print("\n" + "=" * 60)
        print("WEATHER RESOURCE")
        print("=" * 60)

        weather_result = await client.read_resource(
            "weather://Barcelona"
        )

        for content in weather_result.contents:
            if isinstance(content, TextResourceContents):
                print(content.text)


        # ---------------------------------
        # 7. PROMPT
        # ---------------------------------

        print("\n" + "=" * 60)
        print("PROMPT")
        print("=" * 60)

        prompt_result = await client.get_prompt(
            "plan_vacation",
            {
                "destination": "Barcelona",
                "days": "7"
            }
        )

        for message in prompt_result.messages:
            if isinstance(message.content, TextContent):
                print(message.content.text)


await complete_demo()

CONNECTED TO MCP SERVER
Protocol: 2026-07-28
Server: name='Travel MCP Server' title=None version='' description=None website_url=None icons=None

AVAILABLE TOOLS
- add
- search_flights
- book_hotel

CALLING TOOL
Available flights found from Bangalore to Barcelona

RESOURCES
- travel://preferences

TRAVEL PREFERENCES

Preferred airline: Emirates
Preferred seat: Window
Preferred class: Economy
Preferred hotel: 4 Star


WEATHER RESOURCE


  + Exception Group Traceback (most recent call last):
  |   File "d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\IPython\core\interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\Sunny\AppData\Local\Temp\ipykernel_34848\3652573681.py", line 121, in <module>
  |     await complete_demo()
  |   File "C:\Users\Sunny\AppData\Local\Temp\ipykernel_34848\3652573681.py", line 3, in complete_demo
  |     async with Client(mcp) as client:
  |   File "C:\Users\Sunny\AppData\Roaming\uv\python\cpython-3.11.13-windows-x86_64-none\Lib\contextlib.py", line 745, in __aexit__
  |     raise exc_details[1]
  |   File "C:\Users\Sunny\AppData\Roaming\uv\python\cpython-3.11.13-windows-x86_64-none\Lib\contextlib.py", line 728, in __aexit__
  |     cb_suppress = await cb(*exc_details)
  |                   ^^^^^^^^^^^^^^^^^^^^^^
  |   File "d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\sit

                    MCP SERVER
                        │
        ┌───────────────┼───────────────┐
        ↓               ↓               ↓
      TOOLS          RESOURCES        PROMPTS
        │               │               │
        ↓               ↓               ↓
    DO ACTION        READ DATA       GET TEMPLATE
        ↑               ↑               ↑
        └──────────── MCP CLIENT ────────┘

Server = a program/service that receives requests and provides some capability.

Browser / Client
      ↓
GET /index.html
      ↓
Web Server
      ↓
index.html
      ↓
Browser displays page

server ki capability hai:

Web pages/files provide karna.

For example:

Chrome
  ↓ HTTP request
Nginx / Flask / Node server
  ↓
HTML / JSON


MCP Server
AI Application
      ↓
MCP Client
      ↓
MCP Server

MCP server ki capability HTML dena nahi hai.

Instead, wo provide karta hai:

Tools
Resources
Prompts

So:

Web Server
→ Web pages / APIs provide karta hai

MCP Server
→ AI ke liye Tools / Resources / Prompts provide karta hai


web API me:

Client
  ↓
GET /weather/bangalore
  ↓
Server
  ↓
{"temperature": 28}

MCP me conceptually:

AI Application
      ↓
MCP Client
      ↓
weather://Bangalore
      ↓
MCP Server
      ↓
Weather information

Find flights from Bangalore to Dubai.

Flow:

USER
 ↓
AI Application
 ↓
LLM understands:
"I need search_flights tool"
 ↓
MCP Client
 ↓
Travel MCP Server
 ↓
search_flights(
    "Bangalore",
    "Dubai"
)
 ↓
Result
 ↓
MCP Client
 ↓
LLM
 ↓
USER

Exactly waise hi jaise:

Browser
 ↓
Server API endpoint
 ↓
Backend logic
 ↓
Response

AWS EC2 machine
Azure VM
Physical computer

Server ek role hai.

For example:

Your Laptop
│
├── VS Code
│     ↓
│   MCP Client
│
└── Python Process
      ↓
    MCP Server

Dono same laptop par chal sakte hain.

STDIO transport me exactly ye common hai:

VS Code / Claude
       ↓
    MCP Client
       ↓
      STDIO
       ↓
Python MCP Server

No internet required.

Remote bhi ho sakta hai

Production me:

Your AI Application
       ↓
    MCP Client
       ↓
Streamable HTTP
       ↓
Remote MCP Server
       ↓
Company Database / APIs

For example:

ChatGPT
   ↓
MCP Client
   ↓
Company MCP Server
   ↓
 ┌─────────────┐
 │ CRM         │
 │ Database    │
 │ Jira        │
 │ APIs        │
 └─────────────┘
 
 Ek aur analogy

Suppose restaurant hai.

Customer
   ↓
Waiter
   ↓
Kitchen

MCP me:

AI / Host
   ↓
MCP Client
   ↓
MCP Server

Kitchen ke paas:

Pizza banana
Burger banana
Coffee banana

MCP Server ke paas:

search_flights()
send_email()
query_database()

Kitchen ko server bol sakte ho because it provides services/capabilities.